# Agentic RAG Router — 01: Build the oracle routing dataset

Every benchmark question is answered by all five specialist agents. For each answer we record correctness and the exact cost (LLM calls, tokens, FLOPs, time). The **oracle label** of a question is the cheapest agent that answered it correctly. These labels are what the neural router learns to predict.

| Agent | Generator | Strategy |
|---|---|---|
| `A_direct_small` | Qwen2.5-1.5B-Instruct | no retrieval |
| `B_direct_large` | Qwen2.5-7B-Instruct | no retrieval |
| `C_rag_small` | Qwen2.5-1.5B-Instruct | retrieve once (top-5) |
| `D_rag_large` | Qwen2.5-7B-Instruct | retrieve once (top-5) |
| `E_iterative_large` | Qwen2.5-7B-Instruct | up to 3 reason/retrieve steps |

A prediction is **correct** if it matches exactly, contains the gold answer, or has token F1 ≥ 0.6.

Retrieval: `intfloat/multilingual-e5-base` (768-d) over one pooled corpus.
Questions: NQ-Open, SQuAD v1.1, HotpotQA (distractor), MuSiQue-Ans (validation splits).

**Kaggle settings:** Accelerator = *GPU T4 x2*, Internet = *On*.
Run with `RUN_MODE = "smoke"` first (~15 min), then switch to `"full"`. The run is resumable: if the session stops, run all cells again and finished questions are skipped.

In [1]:
import os, sys, subprocess, json, time, platform
from pathlib import Path

REPO_URL = "https://github.com/MHKhan50/agentic-rag-router"
REPO_DIR = Path("/kaggle/working/agentic-rag-router") if Path("/kaggle").exists() else Path("agentic-rag-router").resolve()
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
sys.path.insert(0, str(REPO_DIR / "src"))

import numpy as np, pandas as pd, torch, transformers, datasets
print("python", platform.python_version(), "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| datasets", datasets.__version__)
assert torch.cuda.is_available(), "Turn on a GPU accelerator (Settings > Accelerator > GPU T4 x2)"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}", p.name, f"{p.total_memory/1e9:.1f} GB")
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo commit", COMMIT)

Cloning into '/kaggle/working/agentic-rag-router'...


python 3.13.16 | torch 2.11.0+cu130 | transformers 5.20.2 | datasets 4.8.5
cuda:0 Tesla T4 15.6 GB
cuda:1 Tesla T4 15.6 GB
repo commit 5712dd7


In [2]:
from agentic_rag_router.config import PipelineConfig

RUN_MODE = "smoke"   # "smoke" -> 10 questions per dataset; "full" -> 600 per dataset

cfg = PipelineConfig.smoke() if RUN_MODE == "smoke" else PipelineConfig()
OUT = Path("/kaggle/working" if Path("/kaggle").exists() else ".") / f"oracle_{RUN_MODE}"
OUT.mkdir(parents=True, exist_ok=True)
print(OUT, cfg.n_per_dataset)

/kaggle/working/oracle_smoke {'nq_open': 10, 'squad': 10, 'hotpotqa': 10, 'musique': 10}


In [3]:
from agentic_rag_router.data import load_questions, build_corpus
from agentic_rag_router.pipeline import write_jsonl

questions = load_questions(cfg.n_per_dataset, cfg.seed)
corpus = build_corpus(questions)
print(len(questions), "questions |", len(corpus), "passages in the pooled corpus")
print(pd.Series([q.dataset for q in questions]).value_counts().to_string())

write_jsonl(OUT / "questions.jsonl", (
    {"qid": q.qid, "dataset": q.dataset, "question": q.question, "answers": q.answers,
     "gold_pids": q.gold_pids, "meta": q.meta} for q in questions))
write_jsonl(OUT / "corpus.jsonl", ({"pid": p.pid, "title": p.title, "text": p.text} for p in corpus))

README.md:   0%|          | 0.00/8.77k [00:00<?, ?B/s]

nq_open/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.46MB            

nq_open/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

nq_open/validation-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B /  214kB            

nq_open/validation-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87925 [00:00<?, ? examples/s]

... (20 outputs left out)

musique_ans_v1.0_dev.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19938 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2417 [00:00<?, ? examples/s]

40 questions | 302 passages in the pooled corpus
nq_open     10
squad       10
hotpotqa    10
musique     10


In [4]:
from agentic_rag_router.embedder import E5Embedder
from agentic_rag_router.retrieval import DenseRetriever

embedder = E5Embedder(cfg.embedding_model, device="cuda:0")

pemb_path = OUT / "passage_emb.npy"
if pemb_path.exists() and np.load(pemb_path, mmap_mode="r").shape[0] == len(corpus):
    passage_emb = np.load(pemb_path)
else:
    t0 = time.time()
    passage_emb = embedder.encode_passages([p.render() for p in corpus])
    np.save(pemb_path, passage_emb)
    print(f"embedded {len(corpus)} passages in {time.time()-t0:.0f}s")
retriever = DenseRetriever(corpus, passage_emb)

# Router input features: one 768-d query embedding per question.
query_emb = embedder.encode_queries([q.question for q in questions])
np.save(OUT / "query_emb.npy", query_emb)
(OUT / "query_emb_qids.json").write_text(json.dumps([q.qid for q in questions]))
print("query embeddings", query_emb.shape)

# Retrieval sanity check: how often the question's own gold passage is in the top 5.
hits = retriever.search(query_emb, 5)
rec = [len(set(q.gold_pids) & {p.pid for p in h}) / len(q.gold_pids) for q, h in zip(questions, hits) if q.gold_pids]
print(f"gold-passage recall@5 (single retrieval): {np.mean(rec):.3f}")

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedded 302 passages in 4s
query embeddings (40, 768)
gold-passage recall@5 (single retrieval): 0.853


In [5]:
from agentic_rag_router.pipeline import run_agents

t0 = time.time()
runs = run_agents(questions, cfg, retriever, embedder.encode_queries, OUT)
print(f"agents finished in {(time.time()-t0)/60:.1f} min")

[load] Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[B_direct_large] 40/40  0.78s/q  eta 0.0 min
[D_rag_large] 40/40  5.22s/q  eta 0.0 min
[E_iterative_large] 40/40  7.76s/q  eta 0.0 min
agents finished in 12.4 min


In [6]:
from agentic_rag_router.oracle import score_runs, build_oracle, assign_splits, summarise

long = score_runs(questions, runs)
oracle = build_oracle(long, cfg.correct_f1_threshold)
oracle["split"] = assign_splits(oracle, cfg.split_fractions, cfg.seed)
order = oracle.attrs["agent_order"]
print("agent cost order (by measured FLOPs):", order)

print("\n== Accuracy and cost per agent (all questions) ==")
display(summarise(long, oracle, cfg.correct_f1_threshold))

from agentic_rag_router.oracle import add_correct
print("\n== Accuracy (correct) per agent per dataset ==")
display(add_correct(long, cfg.correct_f1_threshold).pivot_table(index="dataset", columns="agent", values="correct", aggfunc="mean")[order].round(3))

print("\n== Oracle label distribution per dataset ==")
display(pd.crosstab(oracle["dataset"], oracle["oracle_agent"])[[a for a in order if a in set(oracle["oracle_agent"])]])
print(f"questions no agent answered: {oracle['none_correct'].mean():.1%}")
print(oracle["split"].value_counts().to_string())

long.to_csv(OUT / "agent_results_long.csv", index=False)
oracle.to_csv(OUT / "oracle_labels.csv", index=False)

agent cost order (by measured FLOPs): ['A_direct_small', 'B_direct_large', 'C_rag_small', 'D_rag_large', 'E_iterative_large']

== Accuracy and cost per agent (all questions) ==


,correct,em,f1,acc,llm_calls,tokens,seconds,gflops
agent,,,,,,,,
A_direct_small,0.125,0.125,0.1570,0.125,1.000,73.000,0.1671,244.9103
B_direct_large,0.250,0.200,0.2679,0.200,1.000,73.000,0.7764,1204.0290
C_rag_small,0.375,0.350,0.3935,0.350,1.000,706.225,1.3161,2199.7929
D_rag_large,0.400,0.325,0.4060,0.350,1.000,706.225,5.2216,10845.0187
E_iterative_large,0.500,0.425,0.4792,0.450,1.725,889.000,7.7547,13795.6893
ORACLE_ROUTER,0.525,0.425,0.4984,0.450,1.625,725.625,5.7730,9979.2484



== Accuracy (correct) per agent per dataset ==


agent,A_direct_small,B_direct_large,C_rag_small,D_rag_large,E_iterative_large
dataset,,,,,
hotpotqa,0.3,0.6,0.6,0.6,0.7
musique,0.1,0.1,0.2,0.3,0.5
nq_open,0.0,0.0,0.0,0.0,0.0
squad,0.1,0.3,0.7,0.7,0.8



== Oracle label distribution per dataset ==


oracle_agent,A_direct_small,B_direct_large,C_rag_small,D_rag_large,E_iterative_large
dataset,,,,,
hotpotqa,3,4,1,0,2
musique,1,1,0,1,7
nq_open,0,0,0,0,10
squad,1,2,4,1,2


questions no agent answered: 47.5%
split
train    28
val       8
test      4


In [7]:
# Spot-check: predictions next to gold answers for a few questions per dataset.
gold = {q.qid: q.answers[:3] for q in questions}
sample = long[long["qid"].isin(oracle.groupby("dataset").head(2)["qid"])]
view = sample.pivot(index="qid", columns="agent", values="prediction")[order]
view.insert(0, "gold", view.index.map(gold))
pd.set_option("display.max_colwidth", 60)
display(view)

agent,gold,A_direct_small,B_direct_large,C_rag_small,D_rag_large,E_iterative_large
qid,,,,,,
hotpotqa-5a721dfd55429971e9dc928b,[Thorgan Ganael Francis Hazard],Kevin,Thorgan Hazard,Eden,Thorgan Hazard,Thorgan Hazard
hotpotqa-5a758ce05542992db947367c,[Jenny Bae],Lionel Richie,Jenny Bae,Jenny Bae,Lionel Richie,Jenny Bae
musique-2hop__107601_110222,[10 June 1819],1920,"November 13, 1949",1865,1863,10 June 1819
musique-2hop__388467_110882,[1 January 1986],1946,"June 14, 1972",1978,Not mentioned,1 January 1986
nq_open-1057,"[83, more than 80]",105,104,100,not mentioned,109
nq_open-1591,[Dr. Hartwell Carver],john pope and zenas root,Theodore Judah,Not mentioned in the given passages,Not mentioned,Not mentioned
squad-56d98b33dc89441400fdb53c,[three],10,3,three,3,3
squad-56e0c2bc231d4119001ac389,"[being drafted into the Austro-Hungarian Army, being dra...",assassination attempt,war,"Running away to Tomingaj, near Gračac",to evade being drafted into the Austro-Hungarian Army,To evade being drafted into the Austro-Hungarian Army


In [8]:
# The router consumes exactly these features and labels. Quick shape/forward check.
from agentic_rag_router.router import DynamicNeuralRouter

qid_index = {qid: i for i, qid in enumerate(json.loads((OUT / "query_emb_qids.json").read_text()))}
X = torch.tensor(query_emb[[qid_index[q] for q in oracle["qid"]]])
y = torch.tensor(oracle["oracle_label"].values)
router = DynamicNeuralRouter(d_model=X.shape[1], num_agents=len(order))
logits = router(X)
print("features", tuple(X.shape), "| labels", tuple(y.shape), "| logits", tuple(logits.shape))
print("untrained loss", torch.nn.functional.cross_entropy(logits, y).item(), "(~ln K expected)", f"K={len(order)}")

features (40, 768) | labels (40,) | logits (40, 5)
untrained loss 1.7101577520370483 (~ln K expected) K=5


In [9]:
import shutil
manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repo_commit": COMMIT,
    "run_mode": RUN_MODE,
    "config": cfg.to_dict(),
    "agent_cost_order": order,
    "n_questions": len(questions),
    "n_passages": len(corpus),
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "datasets": datasets.__version__},
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive(str(OUT), "zip", OUT)
print("saved", archive)
print(sorted(p.name for p in OUT.iterdir()))

saved /kaggle/working/oracle_smoke.zip
['agent_results_long.csv', 'corpus.jsonl', 'manifest.json', 'oracle_labels.csv', 'passage_emb.npy', 'query_emb.npy', 'query_emb_qids.json', 'questions.jsonl', 'runs']


### Outputs (in `oracle_<mode>/`)
- `questions.jsonl`, `corpus.jsonl` — sampled questions and the pooled passage corpus
- `runs/<agent>.jsonl` — raw agent outputs with full cost accounting and step traces
- `agent_results_long.csv` — one row per (question, agent): EM, F1, Acc, cost
- `oracle_labels.csv` — one row per question: oracle agent, per-agent correctness/FLOPs, split
- `query_emb.npy` + `query_emb_qids.json` — 768-d router features
- `manifest.json` — config, versions, hardware, repo commit

**Next:** train the router and the baselines on `oracle_labels.csv` + `query_emb.npy` (notebook 02).